In [ ]:
import pandas as pd
import numpy as np
from cleantext import clean
import re
from transformers import XLNetTokenizer, XLNetForSequenceClassification, TrainingArguments, Trainer, pipeline
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import datasets 
import evaluate
import random

## Preprocess our data

In [ ]:
# data should be saved in a folder called 'emotions_data' which is saved in the same place as your notebook
data_train = pd.read_csv('./emotions_data/emotion-labels-train.csv') 
data_test = pd.read_csv('./emotions_data/emotion-labels-test.csv')
data_val = pd.read_csv('./emotions_data/emotion-labels-val.csv')


In [ ]:
data_train.head()

In [ ]:
# Combines three separate datasets (train, test, validation) into a single DataFrame. 
# ignore_index=True resets the row numbers so they run continuously from 0 instead of overlapping.
data = pd.concat([data_train, data_test, data_val], ignore_index=True)

In [ ]:
# Creates a new column text_clean by running each row in the text column through the clean() function from cleantext. 
# no_emoji=True strips out all emojis from the text.
data['text_clean'] = data['text'].apply(lambda x: clean(x, no_emoji=True))

In [ ]:
# Updates text_clean by removing all Twitter/social media mentions like @username from each row. 
# The regex @[^\s]+ matches @ followed by any non-space characters.
data['text_clean'] = data['text_clean'].apply(lambda x: re.sub('@[^\s]+', '', x))

In [ ]:
# Displays the first 20 rows of the DataFrame so you can visually verify the cleaning worked correctly.
data.head(20)

In [ ]:
print(data_train.columns.tolist())
print(data_test.columns.tolist())
print(data_val.columns.tolist())

In [ ]:
# This creates a bar chart showing the distribution of labels in your dataset. Here's what each part does:
# .data['label'] — selects the label column from your DataFrame
# .value_counts() — counts how many times each label appears (e.g. positive: 4000, negative: 3500, neutral: 2000)
# .plot(kind="bar") — plots those counts as a bar chart
# It's a quick way to check if your dataset is balanced or imbalanced across classes — an important thing to know before training a model.
data['label'].value_counts().plot(kind="bar")

In [ ]:
# The following is called undersampling. It prevents the model from being biased toward the majority class by trimming all classes down to match 
# the smallest one. After these two lines, every label will have exactly the same number of rows.
# These two lines balance the dataset so every label has the same number of samples. Here's the breakdown:
# Line 1:
# data.groupby('label') — groups the DataFrame by label (e.g. positive, negative, neutral)
# Line 2:
# g.size().min() — finds the smallest class size (e.g. if positive=4000, negative=3500, neutral=2000, this returns 2000)
# x.sample(g.size().min()) — randomly samples that many rows from each group, so all classes end up with the same count
# reset_index(drop=True) — resets row numbers cleanly
# pd.DataFrame(...) — wraps the result back into a DataFrame 
# g = data.groupby('label')
# data = pd.DataFrame(g.apply(lambda x: x.sample(g.size().min()).reset_index(drop=True)))
min_count = data['label'].value_counts().min()
data = pd.concat([
    data[data['label'] == label].sample(min_count) 
    for label in data['label'].unique()
]).reset_index(drop=True)
# list all columns labels
print(data.columns.tolist())

In [ ]:
# Same line as before — but now it serves as a verification step.
# After the undersampling you just did, this bar chart should now show all labels with equal height bars, confirming that the balancing worked correctly.
# Compare it to the first bar chart you plotted earlier — that one likely showed uneven bars, while this one should look uniform across all classes.
data['label'].value_counts().plot(kind="bar")

In [ ]:
# This creates a new column label_int that converts text labels into numbers. For example:
# "negative" → 0
# "neutral" → 1
# "positive" → 2
# Here's the breakdown:
# LabelEncoder() — creates a label encoder object from scikit-learn
# .fit_transform(data['label']) — learns the unique labels and immediately converts them to integers in one step
# data['label_int'] — stores the numeric result as a new column

# Why this is needed — machine learning models can't work with raw text labels like "positive" or "negative". 
# They require numbers. This numeric column will be used as the target variable during model training.
data['label_int'] = LabelEncoder().fit_transform(data['label'])

In [ ]:
# train_test_split is called twice to create three dataset splits:
#
# Line 1: Split full dataset
#   - 80% → train_split (training + validation)
#   - 20% → test_split  (final evaluation, untouched during training)
#
# Line 2: Split train_split further
#   - 90% of train_split → train_split (model learns from this)
#   - 10% of train_split → val_split   (monitors performance, catches overfitting)
#
# Final proportions of original dataset:
#   - Train      : 72%  (0.8 × 0.9)
#   - Validation :  8%  (0.8 × 0.1)
#   - Test       : 20%
#
# Why three splits?
#   - Train      : model learns patterns from this data
#   - Validation : used during training to tune and monitor for overfitting
#   - Test       : final unbiased evaluation after training is complete

train_split, test_split = train_test_split(data, train_size=0.8)
train_split, val_split  = train_test_split(train_split, train_size=0.9)

In [ ]:
# Verify the split sizes after train_test_split:
#
# Total rows = 6132 (4414 + 1227 + 491)
#
# Split      | Actual | Expected % | Actual %
# -----------+--------+------------+---------
# Train      |  4414  |    72%     |  71.9% ✅
# Test       |  1227  |    20%     |  20.0% ✅
# Validation |   491  |     8%     |   8.0% ✅
#
# Splits landed exactly where expected.
# Dataset is correctly divided and ready for model training.

print(len(train_split))
print(len(test_split))
print(len(val_split))

In [ ]:
# Create clean DataFrames for each split, keeping only the two columns
# needed for model training: numeric label and cleaned text.
#
# train_df and test_df each contain:
#   - "label" : integer-encoded labels (from label_int column)
#   - "text"  : cleaned text (from text_clean column)
#
# .values converts each column to a NumPy array before loading
# into the new DataFrame — ensures clean indexing with no carry-over
# from the original split indices.

train_df = pd.DataFrame({
    "label": train_split.label_int.values,
    "text": train_split.text_clean.values
})
test_df = pd.DataFrame({
    "label": test_split.label_int.values,
    "text": test_split.text_clean.values
})

In [ ]:
# Convert pandas DataFrames into HuggingFace Dataset objects.
#
# HuggingFace's Trainer requires data in Dataset format, not pandas DataFrames.
# datasets.Dataset.from_dict() performs this conversion.
#
# Benefits of HuggingFace Dataset format over pandas DataFrame:
#   - Compatible with HuggingFace Trainer and tokenizer .map() function
#   - Memory efficient — supports Arrow format for large datasets
#   - Built-in batching and shuffling during training

train_df = datasets.Dataset.from_dict(train_df)
test_df = datasets.Dataset.from_dict(test_df)

In [ ]:
# Combine train and test Dataset objects into a single DatasetDict.
#
# DatasetDict is a dictionary-like container that holds multiple
# dataset splits under named keys:
#   - "train" : training data (4414 rows)
#   - "test"  : test data    (1227 rows)
#
# Benefits of DatasetDict:
#   - Single object to pass around instead of separate variables
#   - HuggingFace Trainer can directly access splits by name
#   - Enables consistent tokenization across all splits in one .map() call

dataset_dict = datasets.DatasetDict({"train": train_df, "test": test_df})

In [ ]:
# Calling dataset_dict without print() or any method simply displays
# a summary of the DatasetDict in Jupyter — showing:
#   - Split names  : train, test
#   - Number of rows per split
#   - Column names and their data types (features)
#
# Example output:
# DatasetDict({
#     train: Dataset({
#         features: ['label', 'text'],
#         num_rows: 4414
#     })
#     test: Dataset({
#         features: ['label', 'text'],
#         num_rows: 1227
#     })
# })
#
# Useful as a quick sanity check before moving on to tokenization.

dataset_dict

## Create embeddings

In [ ]:
tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding = "max_length", max_length = 128, truncation=True)

In [ ]:
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

In [ ]:
tokenized_datasets

In [ ]:
print(tokenized_datasets['train']['text'][0])

In [ ]:
print(tokenized_datasets['train']['input_ids'][0])

In [ ]:
tokenizer.decode(5)

In [ ]:
print(tokenized_datasets['train']['token_type_ids'][0])

In [ ]:
print(tokenized_datasets['train']['attention_mask'][0])

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

## Fine tune our model

In [ ]:
# sets a constant that stores the number of unique classes in your dataset 
# It tells the model how many output neurons to create in its final classification layer — one per label. 
# Using a named constant like NUM_LABELS is good practice since you only need to change it in one place if your dataset changes.
NUM_LABELS = 4

In [ ]:
model = XLNetForSequenceClassification.from_pretrained('xlnet-base-cased', 
                                                       num_labels=NUM_LABELS, 
                                                       id2label={0: 'anger', 1: 'fear', 2: 'joy', 3: 'sadness'})

In [ ]:
metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", eval_strategy="epoch", num_train_epochs=3)

In [ ]:
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics)

In [ ]:
trainer.train()

## Evaluate model

In [ ]:
trainer.evaluate()

In [ ]:
model.save_pretrained("fine_tuned_model")

In [ ]:
fine_tuned_model = XLNetForSequenceClassification.from_pretrained("fine_tuned_model")

In [ ]:
clf = pipeline("text-classification", fine_tuned_model, tokenizer=tokenizer)

In [ ]:
rand_int = random.randint(0, len(val_split))
print(val_split['text_clean'][rand_int])
answer = clf(val_split['text_clean'][rand_int], top_k=None)
print(answer)